# 🧠 Notebook 2 — Embedding Exploration

**DocuMind AI Portfolio Project**

This notebook covers:
1. What are embeddings?
2. OpenAI vs HuggingFace embeddings
3. Embedding sample documents
4. PCA/t-SNE visualisation of embedding space
5. Semantic similarity examples
6. FAISS index creation demo
7. Similarity search demo

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','text.color':'#e6edf3','axes.labelcolor':'#e6edf3','xtick.color':'#e6edf3','ytick.color':'#e6edf3'})
print('✅ Ready')

## 1. What Are Embeddings?

Embeddings convert text into dense numerical vectors that capture **semantic meaning**.
Similar texts produce similar vectors — enabling semantic search without keyword matching.

Key properties:
- OpenAI `text-embedding-ada-002`: 1536-dimensional, best quality
- HuggingFace `all-MiniLM-L6-v2`: 384-dimensional, free & fast
- Cosine similarity measures the angle between vectors (1.0 = identical, 0.0 = unrelated)

In [ ]:
from src.embedding_engine import EmbeddingEngine
from src.document_processor import load_config

config = load_config('config/config.yaml')
engine = EmbeddingEngine(config)

# Run smoke test
ok = engine.test_embeddings()
print(f'Embedding engine ready: {ok}')
print(f'Provider: {engine.provider}')
print(f'Model: {engine.model_name}')
print(f'Dimension: {engine.dimension}')

## 2. Embedding Sample Texts

In [ ]:
sample_texts = [
    'The company provides 18 days of annual leave per year.',
    'Employees are entitled to 18 paid vacation days annually.',
    'Revenue grew by 32% to reach ₹45 Crore in FY 2024.',
    'The financial performance was strong with a 32% year-over-year increase.',
    'Python 3.10 or higher is required for installation.',
    'Docker containers are used for deployment.',
    'What is the leave policy?',
    'How many vacation days do I get?',
]

vectors = engine.embed_documents(sample_texts)
print(f'Embedded {len(vectors)} texts')
print(f'Vector dimension: {len(vectors[0])}')

## 3. Semantic Similarity Examples

In [ ]:
import itertools

# Compute pairwise cosine similarities
n = len(sample_texts)
sim_matrix = np.zeros((n, n))
for i, j in itertools.product(range(n), range(n)):
    sim_matrix[i, j] = engine.compute_similarity(vectors[i], vectors[j])

# Show interesting pairs
print('Semantic Similarity between text pairs:')
print('-' * 70)
pairs = [
    (0, 1, 'Leave policy paraphrase'),
    (2, 3, 'Revenue paraphrase'),
    (6, 7, 'Question paraphrase'),
    (0, 6, 'Doc-to-query: leave'),
    (2, 6, 'Doc-to-query: mismatch'),
    (4, 5, 'Unrelated tech topics'),
]
for i, j, label in pairs:
    sim = engine.compute_similarity(vectors[i], vectors[j])
    print(f'  [{sim:.4f}] {label}')

## 4. PCA Visualisation of Embedding Space

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
reduced = pca.fit_transform(np.array(vectors))

colors = ['#7C3AED']*2 + ['#06B6D4']*2 + ['#10B981']*2 + ['#EF4444']*2
categories = ['Leave']*2 + ['Finance']*2 + ['Tech']*2 + ['Query']*2

fig, ax = plt.subplots(figsize=(10, 7))
for i, (x, y) in enumerate(reduced):
    ax.scatter(x, y, color=colors[i], s=120, zorder=3)
    ax.annotate(
        sample_texts[i][:35] + '...',
        (x, y), xytext=(8, 4), textcoords='offset points',
        fontsize=8, color=colors[i]
    )
ax.set_title(f'Embeddings in 2D PCA Space (explained var: {pca.explained_variance_ratio_.sum():.1%})', pad=15)
ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()
print('\n💡 Semantically similar texts cluster together in embedding space.')

## 5. FAISS Index Creation & Search Demo

In [ ]:
from src.vector_store import VectorStoreManager
from src.document_processor import DocumentProcessor

processor = DocumentProcessor(config)
vsm = VectorStoreManager(config, engine.embeddings)

# Load and chunk sample documents
raw_docs = processor.load_multiple_documents('data/raw/sample_docs')
chunks = processor.chunk_documents(raw_docs)
chunks = processor.deduplicate_documents(chunks)

print(f'Creating FAISS index from {len(chunks)} chunks ...')
vs = vsm.create_vector_store(chunks)

stats = vsm.get_index_stats()
print(f'\nIndex Stats:')
for k, v in stats.items():
    print(f'  {k}: {v}')

In [ ]:
# Similarity search demo
queries = [
    'annual leave policy',
    'revenue and financial results',
    'Python installation requirements',
]

for q in queries:
    results = vsm.similarity_search(q, k=2)
    print(f'\nQuery: "{q}"')
    for i, doc in enumerate(results, 1):
        print(f'  {i}. {doc.metadata.get("filename","")} | Page {doc.metadata.get("page","N/A")}')
        print(f'     {doc.page_content[:100]}...')

In [ ]:
# MMR vs Similarity search comparison
query = 'employee policies and benefits'
print(f'Query: "{query}"\n')

sim_docs = vsm.similarity_search(query, k=5)
mmr_docs = vsm.mmr_search(query, k=5)

print('Similarity Search results:')
for d in sim_docs:
    print(f'  - {d.metadata.get("filename","")} p.{d.metadata.get("page","")}')

print('\nMMR Search results (more diverse):')
for d in mmr_docs:
    print(f'  - {d.metadata.get("filename","")} p.{d.metadata.get("page","")}')

## Summary

- Embeddings encode semantic meaning in dense vectors
- Semantically similar texts have high cosine similarity
- FAISS enables fast approximate nearest-neighbour search at scale
- MMR search balances relevance AND diversity — reducing redundant results
- DocuMind uses OpenAI ada-002 (paid) or MiniLM (free) depending on `FREE_MODE`